In [3]:
# run autoreload
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json
import sys
import importlib
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src" / "zjet_corrections").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str((repo_root / "src").resolve()))

import zjet_corrections.notebook_utils as nbutils
importlib.reload(nbutils)

# -------------------- Persistence --------------------
CONFIG_FILE = Path(".analysis_widget_config.json")

DEFAULTS = {
    "casa": True,
    "test": False,
    "useDefault": False,
    "executor_mode": "futures",
    "mode": "validation",
    "era": "2016",
    "dataset": "pythia",
    "chunksize": 50000,
    "chunksize_test": 200000,
    "group_mode": "all_in_one",
    "prependstr": "root://xcache/",
    "systematic_profile": "all_syst",
}


def load_config():
    """Load last saved config if available, otherwise return defaults."""
    if CONFIG_FILE.exists():
        try:
            with open(CONFIG_FILE, "r") as f:
                loaded = json.load(f)

            config = DEFAULTS.copy()
            config.update(loaded)
            return config

        except Exception as e:
            print(f"Warning: could not read {CONFIG_FILE}: {e}")
            print("Falling back to defaults.")
    return DEFAULTS.copy()



def save_config(config):
    """Save current config to disk."""
    try:
        with open(CONFIG_FILE, "w") as f:
            json.dump(config, f, indent=2)
    except Exception as e:
        print(f"Warning: could not save config: {e}")



def apply_config_to_widgets(config):
    """Push config values into widgets."""
    casa_w.value = config["casa"]
    test_w.value = config["test"]
    useDefault_w.value = config["useDefault"]
    executor_mode_w.value = config.get("executor_mode", DEFAULTS["executor_mode"])
    mode_w.value = config["mode"]
    era_w.value = config["era"]
    dataset_w.value = config["dataset"]
    chunksize_w.value = config["chunksize"]
    chunksize_test_w.value = config["chunksize_test"]
    group_mode_w.value = config["group_mode"]
    prependstr_w.value = config["prependstr"]
    systematic_profile_w.value = config["systematic_profile"]



def get_config_from_widgets():
    """Pull current widget values into a config dict."""
    return {
        "casa": casa_w.value,
        "test": test_w.value,
        "useDefault": useDefault_w.value,
        "executor_mode": executor_mode_w.value,
        "mode": mode_w.value,
        "era": era_w.value,
        "dataset": dataset_w.value,
        "chunksize": chunksize_w.value,
        "chunksize_test": chunksize_test_w.value,
        "group_mode": group_mode_w.value,
        "prependstr": prependstr_w.value,
        "systematic_profile": systematic_profile_w.value,
    }



def apply_widget_values(save=False, show_output=True):
    """
    Read current widget values and apply them to notebook variables.
    Optionally save them and/or print them.
    """
    global casa, test, mode, era, data, dataset
    global chunksize, chunksize_test, useDefault, executor_mode
    global group_mode, prependstr, systematic_profile
    global systematics_list, jet_systematics_list

    casa = casa_w.value
    test = test_w.value
    mode = mode_w.value
    era = era_w.value
    dataset = dataset_w.value
    data = dataset == "data"
    chunksize = chunksize_w.value
    chunksize_test = chunksize_test_w.value
    useDefault = useDefault_w.value
    executor_mode = executor_mode_w.value
    group_mode = group_mode_w.value
    prependstr = prependstr_w.value
    systematic_profile = systematic_profile_w.value
    systematics_list, jet_systematics_list = nbutils.resolve_systematics(systematic_profile)

    if save:
        save_config(get_config_from_widgets())

    if show_output:
        with output:
            output.clear_output()
            print("Applied settings:")
            print(f"casa = {casa}")
            print(f"test = {test}")
            print(f"mode = {mode}")
            print(f"era = {era}")
            print(f"data = {data}")
            print(f"dataset = {dataset}")
            print(f"systematic_profile = {systematic_profile}")
            print(f"systematics_list = {systematics_list}")
            print(f"jet_systematics_list = {jet_systematics_list}")
            print(f"chunksize = {chunksize}")
            print(f"chunksize_test = {chunksize_test}")
            print(f"useDefault = {useDefault}")
            print(f"executor_mode = {executor_mode}")
            print(f"group_mode = {group_mode}")
            print(f"prependstr = {prependstr}")
            print("output_dir = outputs/")
            if save:
                print(f"\nSaved to: {CONFIG_FILE}")


# Load last saved config (or defaults if none exists)
CONFIG = load_config()

DATASET_OPTIONS = nbutils.DATASET_OPTIONS
ERA_OPTIONS = nbutils.ERA_OPTIONS
MODE_OPTIONS = list(nbutils.MODE_OPTIONS)
if "mass_diagnostic_ntuple" not in MODE_OPTIONS:
    MODE_OPTIONS.append("mass_diagnostic_ntuple")
SYSTEMATIC_PROFILE_OPTIONS = nbutils.SYSTEMATIC_PROFILE_OPTIONS
EXECUTOR_MODE_OPTIONS = nbutils.EXECUTOR_MODE_OPTIONS

if CONFIG["dataset"] not in DATASET_OPTIONS:
    CONFIG["dataset"] = DEFAULTS["dataset"]
if CONFIG["era"] not in ERA_OPTIONS:
    CONFIG["era"] = DEFAULTS["era"]
if CONFIG["mode"] not in MODE_OPTIONS:
    CONFIG["mode"] = DEFAULTS["mode"]
if CONFIG.get("systematic_profile") not in SYSTEMATIC_PROFILE_OPTIONS:
    CONFIG["systematic_profile"] = DEFAULTS["systematic_profile"]
if CONFIG.get("executor_mode") not in EXECUTOR_MODE_OPTIONS:
    CONFIG["executor_mode"] = DEFAULTS["executor_mode"]

# -------------------- Widgets --------------------
casa_w = widgets.Checkbox(value=CONFIG["casa"], description="casa")
test_w = widgets.Checkbox(value=CONFIG["test"], description="test")
useDefault_w = widgets.Checkbox(value=CONFIG["useDefault"], description="useDefault")
executor_mode_w = widgets.Dropdown(
    options=EXECUTOR_MODE_OPTIONS,
    value=CONFIG["executor_mode"],
    description="executor"
)

mode_w = widgets.Dropdown(
    options=MODE_OPTIONS,
    value=CONFIG["mode"],
    description="mode"
)

era_w = widgets.Dropdown(
    options=ERA_OPTIONS,
    value=CONFIG["era"],
    description="era"
)

dataset_w = widgets.Dropdown(
    options=DATASET_OPTIONS,
    value=CONFIG["dataset"],
    description="dataset"
)

systematic_profile_w = widgets.Dropdown(
    options=SYSTEMATIC_PROFILE_OPTIONS,
    value=CONFIG["systematic_profile"],
    description="syst profile"
)

chunksize_w = widgets.IntText(
    value=CONFIG["chunksize"],
    description="chunksize"
)

chunksize_test_w = widgets.IntText(
    value=CONFIG["chunksize_test"],
    description="test chunk"
)

group_mode_w = widgets.Dropdown(
    options=["all_in_one", "per_group"],
    value=CONFIG["group_mode"],
    description="group_mode"
)

prependstr_w = widgets.Text(
    value=CONFIG["prependstr"],
    description="prependstr"
)

run_button = widgets.Button(
    description="Apply settings",
    button_style="success"
)

reset_button = widgets.Button(
    description="Reset to defaults",
    button_style="warning"
)

output = widgets.Output()

# -------------------- Layout --------------------
ui = widgets.VBox([
    widgets.HTML("<h3>Analysis configuration</h3>"),
    widgets.HBox([casa_w, test_w, useDefault_w]),
    executor_mode_w,
    mode_w,
    era_w,
    dataset_w,
    systematic_profile_w,
    chunksize_w,
    chunksize_test_w,
    group_mode_w,
    prependstr_w,
    widgets.HBox([run_button, reset_button]),
    output
])

display(ui)

# -------------------- Button actions --------------------
def on_run_clicked(b):
    apply_widget_values(save=True, show_output=True)



def on_reset_clicked(b):
    apply_config_to_widgets(DEFAULTS)
    apply_widget_values(save=True, show_output=True)

    with output:
        output.clear_output()
        print("Reset all widget values to defaults and applied them.")
        print(f"Saved defaults to: {CONFIG_FILE}")
        print(f"casa = {casa}")
        print(f"test = {test}")
        print(f"mode = {mode}")
        print(f"era = {era}")
        print(f"data = {data}")
        print(f"dataset = {dataset}")
        print(f"systematic_profile = {systematic_profile}")
        print(f"systematics_list = {systematics_list}")
        print(f"jet_systematics_list = {jet_systematics_list}")
        print(f"chunksize = {chunksize}")
        print(f"chunksize_test = {chunksize_test}")
        print(f"useDefault = {useDefault}")
        print(f"executor_mode = {executor_mode}")
        print(f"group_mode = {group_mode}")
        print(f"prependstr = {prependstr}")
        print("output_dir = outputs/")


run_button.on_click(on_run_clicked)
reset_button.on_click(on_reset_clicked)

# -------------------- Auto-apply on cell execution --------------------
apply_widget_values(save=False, show_output=True)


In [5]:
paths = nbutils.get_analysis_paths(repo_root)
HT_BINS = nbutils.HT_BINS

SAMPLES_DATA_DIR = paths.samples_data_dir
SAMPLES_MC_DIR = paths.samples_mc_dir
SAMPLES_BKG_DIR = paths.samples_bkg_dir
SAMPLES_MC_LOCAL_DIR = paths.samples_mc_local_dir

samplePath = nbutils.SamplePath(era)

# In[4]:  -------------------- Imports (do once) --------------------
import os
import time
import pickle
import importlib

import numpy as np
import awkward as ak
import uproot

import coffea
from coffea.nanoevents import NanoAODSchema
from coffea import processor
from IPython.display import Audio

importlib.reload(nbutils)
import dask
dask.config.set({
    "distributed.logging.distributed": "error",
    "distributed.logging.bokeh": "error",
    "distributed.logging.tornado": "error",
})


In [6]:
# In[5]:  -------------------- Helpers --------------------
NanoAODSchema.warn_missing_crossrefs = False

format_time = nbutils.format_time
iter_groups = nbutils.iter_groups
build_fileset_from_txts = nbutils.build_fileset_from_txts
build_backgrounds_fileset = nbutils.build_backgrounds_fileset
build_local_pythia_fileset = nbutils.build_local_pythia_fileset
make_runner = nbutils.make_runner
ensure_client = nbutils.ensure_client
upload_package_if_casa = nbutils.upload_package_if_casa
run_once = nbutils.run_once
save_output = nbutils.save_output
make_output_filename = nbutils.make_output_filename
get_group_tag = nbutils.get_group_tag
ST_FILES = nbutils.ST_FILES


In [7]:
# In[6]:  -------------------- Create client --------------------
client = ensure_client(casa=casa, test=test, useDefault = useDefault, executor_mode=executor_mode)
upload_package_if_casa(client, casa=casa)

Running locally with 1-2 files (test=True)
Using FuturesExecutor without a Dask client.


In [8]:
# In[7]:  -------------------- Build fileset(s), run, and save immediately --------------------
outs = []  # keep multiple outputs if you run multiple groups



def run_save_append(
    fileset,
    i,
    *,
    client=None,
    test=False,
    data=False,
    chunksize=None,
    chunksize_test=None,
):
    out_i = run_once(
        fileset,
        client=client,
        test=test,
        data=data,
        mode=mode,
        systematic_profile=systematic_profile,
        chunksize=chunksize,
        chunksize_test=chunksize_test,
        executor_mode=executor_mode,
    )

    tag = get_group_tag(i, era, group_mode)
    fout = make_output_filename(
        data=data,
        dataset=dataset,
        tag=tag,
        mode=mode,
        test=test,
    )

    save_output(out_i, fout)
    print(f"[{i+1}] Saved: {fout}")

    outs.append(out_i)
    return out_i


if data:
    for i, group in enumerate(iter_groups(samplePath.data, group_mode)):
        fileset = build_fileset_from_txts(
            group,
            SAMPLES_DATA_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            i,
            client=client,
            test=test,
            data=True,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

else:
    if dataset == "pythia":
        for i, group in enumerate(iter_groups(samplePath.pythia, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=True,
                ht_bins=HT_BINS,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "pythia_local":
        fileset = build_local_pythia_fileset(SAMPLES_MC_LOCAL_DIR, era)
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "pythia2":
        fileset = build_fileset_from_txts(
            ["inclusive_UL16NanoAODv9.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "herwig":
        for i, group in enumerate(iter_groups(samplePath.herwig, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=False,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "powheg":
        fileset = build_fileset_from_txts(
            ["powheg_UL18NanoAODv9_inclusive.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "st":
        fileset = build_fileset_from_txts(
            ST_FILES,
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "backgrounds":
        fileset = build_backgrounds_fileset(
            SAMPLES_BKG_DIR,
            prependstr,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    else:
        print(f"Dataset is {dataset} and it is not in the list")


# In[8]:  -------------------- Choose what to keep in `out` --------------------
out = outs[-1] if outs else None


# In[10]:  -------------------- Analysis / plotting zone --------------------
# Keep plotting down here so the run block stays clean.
# (Your existing plotting cells can remain, just moved below this line.)

print(f"Number of group outputs: {len(outs)}")

if client is not None:
    client.close()


Running over: ['pythia_UL16NanoAODv9_HT-100to200', 'pythia_UL16NanoAODv9_HT-200to400', 'pythia_UL16NanoAODv9_HT-400to600', 'pythia_UL16NanoAODv9_HT-600to800', 'pythia_UL16NanoAODv9_HT-800to1200', 'pythia_UL16NanoAODv9_HT-1200to2500', 'pythia_UL16NanoAODv9_HT-2500toInf'] 
Running over test files: ['pythia_UL16NanoAODv9_HT-100to200']
[DEBUG] Registered Histograms dict_keys(['reco_jet_ntuple', 'sumw', 'nev', 'cutflow'])


Output()

Output()

/mnt/extra/wsLinux/zjet_corrections/.venv/lib/python3.14/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/mnt/extra/wsLinux/zjet_corrections/.venv/lib/python3.14/site-packages/coffea/nanoevents/schemas/nanoaod.py:283: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/mnt/extra/wsLinux/zjet_corrections/.venv/lib/python3.14/site-packages/coffea/nanoevents/schemas/nanoaod.py:322: RuntimeWarning: Branch Photon_mass already exists but its values will be replaced with 0.0
  warnings.warn(
/mnt/extra/wsLinux/zjet_corrections/.venv/lib/python3.14/site-packages/coffea/nanoevents/schemas/nanoaod.py:322: RuntimeWarning: Branch Photon_charge already exists but its values will be replaced with 0.0
  warnings.warn(


[DEBUG] Systematics ['nominal']
[DEBUG] Current Mode mass_diagnostic_ntuple
[INFO] Starting processing for dataset: pythia_UL16NanoAODv9_HT-100to200 and file: root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL16NanoAODv9/DYJetsToLL_M-50_HT-100to200_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v2/260000/18D0A087-30BD-FE4E-B447-5F493C2D2794.root
[DEBUG] Total events in chunk: 57804


...

Using TAG AK8PFPuppi


[DEBUG] Jet Corrections Applied
[DEBUG] Available Jet systematics ['nominal']
[DEBUG] Processing jet systematic: nominal
[DEBUG] FatJet pt before correction: [[190], [180], [193], [177], [183], ..., [182], [178], [211, 179], [180], [186]]
How many none in Fatjet.mass before processing inside jmrnom 57178
Genmass inside jmrsf [[], [], [], [], [], [], [], [], [], ..., [], [], [], [], [], [], [], [], []]
How many none in Fatjet.matched_gen inside jmrnom 0
[[], [], [], [], [], [], [], [], [], ..., [], [], [], [], [], [], [], [], []]
How many none in Fatjet inside jmrnom 0
How many none in Fatjet.mass that is being returned inside jmrnom 57181
[DEBUG] FatJet pt after correction: [[205], [193], [221], [187], [213], ..., [196], [188], [189], [196], [213]]


[DEBUG] Sum of this sel is 140
[DEBUG] Len? 140  
[DEBUG] Padded Electron/Muon collections to minimum length 2 per event
[INFO] Lepton weights added
[DEBUG] Total reco events passing all selection: 139
[DEBUG] Total gen events passing all selection: 171
[DEBUG] Total events passing both reco and gen selections: 139
[DEBUG] Total reco events (ee channel) passing all selection: 48
[DEBUG] Total reco events (mm channel) passing all selection: 91
[DEBUG] Weights sample: [0.91262936 0.96968779 0.83134429 0.97860846 1.0565171  0.82654144
 0.81750375 0.69713141 1.00722351 0.93962012]
Now doing mm
JET SYST nominal
[DEBUG] list of systematics ['nominal']
[DEBUG] Processing systematic nominal


Fatjet y  [1.39, -0.34, -0.247, -0.553, 1.38, 2.07, ..., -1.19, 0.92, 1.05, 0.16, -1.37]
Fatjet eta  [1.4, -0.347, -0.266, -0.555, 1.39, 2.08, ..., -1.2, 0.936, 1.06, 0.161, -1.38]
[DEBUG] Len of ptreco 91 mreco 91 syst nominal channel mm dataset pythia_UL16NanoAODv9_HT-100to200
[DEBUG] ptreco sample [205, 217, 190, 187, 210, 211, 225, 228, 192, 234]
[DEBUG] mreco sample [35.6, 43.5, 75.3, 13.2, 31.2, 32.6, 76.6, 29.6, 58.5, 30.2]
[DEBUG] mreco_g sample [16.2, 21.1, 75.3, 3.07, 3.73, 7.01, 76.6, 4.08, 48.4, 1.5]
Now doing ee
JET SYST nominal
[DEBUG] list of systematics ['nominal']
[DEBUG] Processing systematic nominal
Fatjet y  [-1.54, -0.661, -0.964, 1.67, 0.995, ..., -0.999, -0.0324, -0.0249, -1.01]
Fatjet eta  [-1.56, -0.669, -0.977, 1.67, 1.01, ..., -0.874, -1.01, -0.0326, -0.0261, -1.02]
[DEBUG] Len of ptreco 48 mreco 48 syst nominal channel ee dataset pythia_UL16NanoAODv9_HT-100to200
[DEBUG] ptreco sample [201, 223, 194, 208, 234, 212, 203, 228, 229, 196]
[DEBUG] mreco sample [44

Done. time taken 1m 10s
Output written to /mnt/extra/wsLinux/zjet_corrections/outputs/mass_diagnostic_ntuple_pythia_2016_TEST.pkl with size 29.6 kB
[1] Saved: /mnt/extra/wsLinux/zjet_corrections/outputs/mass_diagnostic_ntuple_pythia_2016_TEST.pkl
Number of group outputs: 1


In [9]:
import matplotlib.pyplot as plt

if out is None:
    raise RuntimeError("`out` is not defined. Run the analysis cell first.")

response_hist = out["response_matrix_g"].project("ptreco", "mreco", "ptgen", "mgen")
response_values = np.asarray(response_hist.values())
shape0, shape1, shape2, shape3 = response_values.shape
response_flat = response_values.reshape(shape0 * shape1, shape2 * shape3)

row_sums = response_flat.sum(axis=1, keepdims=True)
response_norm = np.divide(
    response_flat,
    row_sums,
    out=np.zeros_like(response_flat, dtype=float),
    where=row_sums > 0,
)
response_masked = np.ma.masked_where(response_norm == 0, response_norm)

bg = "#0f1117"
fg = "#e6edf3"
grid = "#8b949e"
cmap = plt.cm.cividis.copy()
cmap.set_bad(color=bg, alpha=0)

fig, ax = plt.subplots(figsize=(12, 9), facecolor=bg)
ax.set_facecolor(bg)
im = ax.imshow(
    response_masked,
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=0,
    vmax=1,
)

for y_boundary in np.arange(1, shape0) * shape1 - 0.5:
    ax.axhline(y_boundary, color=grid, lw=0.8, alpha=0.45)
for x_boundary in np.arange(1, shape2) * shape3 - 0.5:
    ax.axvline(x_boundary, color=grid, lw=0.8, alpha=0.45)

ptreco_edges = response_hist.axes["ptreco"].edges
ptgen_edges = response_hist.axes["ptgen"].edges

reco_centers = np.arange(shape0) * shape1 + (shape1 - 1) / 2
reco_labels = [f"{lo:.0f}-{hi:.0f}" for lo, hi in zip(ptreco_edges[:-1], ptreco_edges[1:])]
gen_centers = np.arange(shape2) * shape3 + (shape3 - 1) / 2
gen_labels = [f"{lo:.0f}-{hi:.0f}" for lo, hi in zip(ptgen_edges[:-1], ptgen_edges[1:])]

ax.set_yticks(reco_centers)
ax.set_yticklabels(reco_labels, color=fg)
ax.set_xticks(gen_centers)
ax.set_xticklabels(gen_labels, rotation=45, ha="right", color=fg)
ax.set_xlabel("Gen bins grouped by $p_T$ (mass bins vary fastest)", color=fg)
ax.set_ylabel("Reco bins grouped by $p_T$ (mass bins vary fastest)", color=fg)
ax.set_title("Row-normalized flattened response matrix for response_matrix_g", color=fg)
ax.tick_params(colors=fg)
for spine in ax.spines.values():
    spine.set_color(grid)

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Fraction within each reco bin", color=fg)
cbar.ax.yaxis.set_tick_params(color=fg)
plt.setp(cbar.ax.get_yticklabels(), color=fg)
cbar.outline.set_edgecolor(grid)

plt.tight_layout()
plt.show()

KeyError: 'response_matrix_g'

In [ ]:
for key in out:
    print(key)

In [1]:
out['ptjet_mjet_g_reco']

NameError: name 'out' is not defined

In [ ]:
from zjet_corrections import plot_utils

if 'out' not in globals() or out is None:
    raise RuntimeError("`out` is not defined. Run the analysis cell first.")

if 'reco_jet_ntuple' in out:
    plot_utils.plot_reco_jet_ntuple_raw_mass_map(out, era=era, data=data)
else:
    plot_utils.plot_raw_mass_groomed_vs_ungroomed(out, era=era, data=data)
    plot_utils.plot_raw_mass_overlay(out, era=era, data=data)
